# Week 10: Time-to-Second-Purchase — The Make-or-Break Moment
## Accelerated Failure Time (AFT) Models

**Masterclass:** [Week 10 Deep-Dive](https://balaviswanath-analytics.github.io/analytics_masterclass/week10_second_purchase.html)

**Dataset:** CDNOW transactions (bundled with `lifetimes`)

**What You'll Build:**
1. Compute time-to-second-purchase from transaction data
2. Weibull, Log-Normal, Log-Logistic AFT models
3. AIC model selection tournament
4. Predict median time to second purchase per customer
5. Early intervention triggers

In [ ]:
!pip install lifetimes lifelines pandas matplotlib -q

In [ ]:
import pandas as pd
import numpy as np
from lifetimes.datasets import load_transaction_data
from lifelines import WeibullAFTFitter, LogNormalAFTFitter, LogLogisticAFTFitter
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
print("Libraries loaded.")

---
## Part 1: Compute Time-to-Second-Purchase

In [ ]:
txns = load_transaction_data()
txns['date'] = pd.to_datetime(txns['date'])

# First and second purchase dates per customer
customer_txns = txns.sort_values(['id', 'date']).groupby('id')['date'].apply(list).reset_index()
customer_txns['n_purchases'] = customer_txns['date'].apply(len)
customer_txns['first_purchase'] = customer_txns['date'].apply(lambda x: x[0])

# Customers with 2+ purchases: time to second
has_second = customer_txns[customer_txns['n_purchases'] >= 2].copy()
has_second['second_purchase'] = has_second['date'].apply(lambda x: x[1])
has_second['days_to_2nd'] = (has_second['second_purchase'] - has_second['first_purchase']).dt.days

# Customers with only 1 purchase: censored
one_only = customer_txns[customer_txns['n_purchases'] == 1].copy()
cutoff = txns['date'].max()
one_only['days_to_2nd'] = (cutoff - one_only['first_purchase']).dt.days
one_only['second_purchase'] = pd.NaT

# Combine
aft_df = pd.concat([
    has_second[['id', 'days_to_2nd', 'n_purchases']].assign(event=1),
    one_only[['id', 'days_to_2nd', 'n_purchases']].assign(event=0)
])
# Add first order value
first_order = txns.sort_values('date').groupby('id').first()['spent'].rename('first_order_value')
aft_df = aft_df.merge(first_order, on='id')

# Remove zeros/negatives
aft_df = aft_df[aft_df['days_to_2nd'] > 0]

print(f"Customers: {len(aft_df):,}")
print(f"Made 2nd purchase: {aft_df['event'].sum():,} ({aft_df['event'].mean():.1%})")
print(f"Censored (only 1 purchase): {(1-aft_df['event']).sum():.0f}")
print(f"Median days to 2nd purchase (uncensored): {aft_df.loc[aft_df['event']==1, 'days_to_2nd'].median():.0f}")

---
## Part 2: AFT Model Tournament

AFT models predict *time* directly (not hazard). The coefficient exp(beta) is a **time ratio**: how much longer/shorter a customer takes to reach the second purchase.

In [ ]:
# Prepare for lifelines
aft_df['log_first_order'] = np.log1p(aft_df['first_order_value'])
covs = ['log_first_order']

models = {
    'Weibull': WeibullAFTFitter(),
    'Log-Normal': LogNormalAFTFitter(),
    'Log-Logistic': LogLogisticAFTFitter()
}

results = {}
for name, fitter in models.items():
    fitter.fit(aft_df[covs + ['days_to_2nd', 'event']],
               duration_col='days_to_2nd', event_col='event')
    results[name] = {'AIC': fitter.AIC_, 'BIC': fitter.BIC_}
    print(f"\n=== {name} AFT ===")
    print(f"  AIC: {fitter.AIC_:.1f} | BIC: {fitter.BIC_:.1f}")
    fitter.print_summary(columns=['coef', 'exp(coef)', 'p'])

# Winner
best = min(results, key=lambda x: results[x]['AIC'])
print(f"\n*** Best model by AIC: {best} ***")

In [ ]:
# Predict median time-to-second-purchase
best_model = models[best]
aft_df['predicted_median_days'] = best_model.predict_median(aft_df[covs + ['days_to_2nd', 'event']])

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(aft_df.loc[aft_df['event']==1, 'predicted_median_days'].clip(upper=365),
        bins=40, color='#264653', edgecolor='white', alpha=0.85, label='Made 2nd purchase')
ax.hist(aft_df.loc[aft_df['event']==0, 'predicted_median_days'].clip(upper=365),
        bins=40, color='#c45d3e', edgecolor='white', alpha=0.5, label='Censored')
ax.set_title(f'Predicted Median Days to 2nd Purchase ({best} AFT)', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted Median Days')
ax.set_ylabel('Customers')
ax.legend()
plt.tight_layout()
plt.show()

---
## Exercises

### Exercise 1: AFT Interpretation
If exp(beta) for first_order_value = 0.85, what does this mean in plain English? Write a one-sentence interpretation for a product manager.

### Exercise 2: Critical Window
What percentage of second purchases happen within 30 days? 60 days? 90 days? At what day does the hazard of second purchase drop below 1%?

### Exercise 3: Intervention Design
Customers predicted to take >60 days to second purchase need a nudge email at day 14. How many customers fall in this bucket? What should the email contain?

---
## Key Takeaways
1. **AFT models** predict time directly — more intuitive than hazard ratios
2. **exp(beta)** is a time ratio: 0.85 means 15% faster to second purchase
3. **AIC tournament** picks the best distributional assumption
4. **First order value** predicts second purchase timing